In [1]:
import pandas as pd
from datetime import datetime


def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.Timedelta(days=1)
    start_date_dt = end_date_dt - pd.Timedelta(days=last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

def get_backtest_bounds(target_date: str, last_n_days: int):
    target_dt = pd.to_datetime(target_date)
    end_date_dt = target_dt - pd.offsets.BusinessDay(1)
    start_date_dt = end_date_dt - pd.offsets.BusinessDay(last_n_days)
    return start_date_dt.strftime('%Y-%m-%d'), end_date_dt.strftime('%Y-%m-%d')

# P.S. assume the data is queried
target_date = "2026-05-08"
last_n_days = 30
start_date, end_date = get_backtest_bounds(target_date, last_n_days)

In [2]:
save_dir = "/Users/livardywufianto/Projects/Trading/data/minute_aggs"

# Load Data

In [3]:
import pandas as pd
from typing import List
ticker_list = [
    "AAPL",
    "MU",
]

import pandas as pd

def generate_csv_filepaths(start_date, end_date, save_dir):
    date_series = pd.date_range(start=start_date, end=end_date)    
    keys = []
    base_path = save_dir
    
    for dt in date_series:
        full_date = dt.strftime('%Y-%m-%d')
        key = f"{base_path}/{full_date}.csv.gz"
        keys.append(key)        
    return keys

def generate_csv_filepath(date_str, save_dir):
    dt = pd.to_datetime(date_str)
    base_path = save_dir
    full_date = dt.strftime('%Y-%m-%d')
    key = f"{base_path}/{full_date}.csv.gz"
    return key


def convert_timestamp_to_us_datetime(timestamp: int, unit="ms"):
    """
    Converts a millisecond timestamp to US/Eastern time.

    NYSE/NASDAQ are located at US/Eastern
    """
    return pd.to_datetime(
        timestamp, unit=unit, utc=True
    ).tz_convert('US/Eastern')      
    
def load_csv_by_chunking(
    filepath: str, ticker_list: List[str],
    chunk_size = 100000
):

    chunks = []
    for chunk in pd.read_csv(filepath, chunksize=chunk_size):
        filtered_chunk = chunk[chunk["ticker"].isin(ticker_list)]
        chunks.append(filtered_chunk)

    df = pd.concat(chunks)
    return df

In [4]:
csv_filepaths = generate_csv_filepaths(start_date, end_date, save_dir)

In [5]:
df_ls = []
for csv_filepath in csv_filepaths:
    try:
        df_ls.append(
            load_csv_by_chunking(csv_filepath, ticker_list)
        )
    except: 
        pass
df = pd.concat(df_ls).reset_index(drop=True)
target_df = load_csv_by_chunking(
    generate_csv_filepath(target_date, save_dir), ticker_list
)

In [6]:
import numpy as np

def aggregate_to_hour(df):
    df["us_datetime"] = df["window_start"].apply(
        lambda x: convert_timestamp_to_us_datetime(x, unit="ns"))
    df["us_date"] = df["us_datetime"].dt.date
    df["hour"] = df["us_datetime"].dt.hour
    agg_logic = {
        "open": "first",    # The first price of the hour
        "high": "max",      # The highest price in that hour
        "low": "min",       # The lowest price in that hour
        "close": "last",    # The final price of the hour
        "volume": "sum",    # Total volume traded in that hour
        "transactions": "sum" # Total count of trades in that hour
    }
    
    hour_df = df.groupby(
        ["ticker", "us_date", "hour"]
    ).agg(agg_logic).reset_index()
    return hour_df



# Volume Check

In [13]:
apply_log_norm = False
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)
if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
merge_df[merge_df["z_score"].abs() > threshold]

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
24,MU,2026-05-08,12,721.845,742.1499,721.8450,730.03,9.237463e+06,204921,4.427035e+06,1.756601e+06,2.738487
28,MU,2026-05-08,16,746.810,749.8800,720.6147,744.90,1.144602e+06,14112,5.164031e+05,3.068636e+05,2.047159


# Log Volume

In [14]:
apply_log_norm = True
threshold = 2

hour_df = aggregate_to_hour(df)
target_hour_df = aggregate_to_hour(target_df)
if apply_log_norm:
    hour_df["volume"] = np.log1p(hour_df["volume"])
    target_hour_df["volume"] = np.log1p(target_hour_df["volume"])
    
volume_df = hour_df.groupby(["ticker", "hour"]).agg({
    "volume": ["mean", "std"]
})
volume_df.columns = [f"{col}_{stat}" for col, stat in volume_df.columns]
volume_df = volume_df.reset_index()

merge_df = pd.merge(target_hour_df, volume_df, on=["ticker", "hour"])
merge_df["z_score"] = (merge_df["volume"] - merge_df["volume_mean"]) / merge_df["volume_std"]
merge_df[merge_df["z_score"].abs() > threshold]

,ticker,us_date,hour,open,high,low,close,volume,transactions,volume_mean,volume_std,z_score
24,MU,2026-05-08,12,721.845,742.1499,721.845,730.03,16.038778,204921,15.236942,0.360206,2.226048
